# 5. Pre-registered boiling-point tests (Section 6.4, Table 5)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Singati2/LOYOLA-PAPER/blob/main/notebooks/05_boiling_points.ipynb)

Recomputes the within-fold baselines of Table 5 and the 200-permutation check (about 5 minutes). The seed medians come from the committed run; set `RUN_SLOW = True` to recompute them (about 30 minutes) and the exploratory selection-aware resampling (about 30 minutes).

Every check prints the value in the paper next to the recomputed value. The last cell fails if any of them differ.

In [1]:
# Setup: runs inside the repository, or on Google Colab (clones the repository first).
import os, sys, subprocess
if os.path.exists("../verify.py"):
    os.chdir("..")
elif not os.path.exists("verify.py"):
    if not os.path.exists("LOYOLA-PAPER"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Singati2/LOYOLA-PAPER.git"], check=True)
    os.chdir("LOYOLA-PAPER")
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "pandas", "scikit-learn"], check=True)
ROOT = os.getcwd()
for p in (ROOT, os.path.join(ROOT, "external"), os.path.join(ROOT, "external", "bp"), os.path.join(ROOT, "structural")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("working directory:", ROOT)

working directory: /Users/ganeshshiwakoti/LOYOLA-PAPER


In [2]:
import re, csv, time
import numpy as np, pandas as pd
RESULTS = []

def check(what, paper, got, tol=0.0):
    """Record one comparison between a value printed in the paper and the recomputed value."""
    if isinstance(paper, (int, float)) and not isinstance(paper, bool):
        ok = abs(float(paper) - float(got)) <= tol + 1e-12
    else:
        ok = paper == got
    RESULTS.append({"check": what, "paper": paper, "recomputed": got, "status": "OK" if ok else "MISMATCH"})
    print(f"{'OK      ' if ok else 'MISMATCH'} {what}: paper {paper}, recomputed {got}")
    return ok

def run(cmd, tail=25):
    """Run a repository script, show the end of its output and require exit status 0."""
    t = time.time()
    env = {k: v for k, v in os.environ.items() if k != "MPLBACKEND"}   # Jupyter's inline backend must not leak into scripts
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    out = (r.stdout + r.stderr).strip().splitlines()
    print("\n".join(out[-tail:]))
    print(f"[exit {r.returncode}, {time.time() - t:.0f} s]")
    RESULTS.append({"check": f"script: {cmd}", "paper": "exit 0", "recomputed": f"exit {r.returncode}",
                    "status": "OK" if r.returncode == 0 else "MISMATCH"})
    return r.stdout

In [3]:
RUN_SLOW = False

In [4]:
import bp_tests as T
import baselines as B
rows, Pall, y, z = T.load()
dec = [i for i, r in enumerate(rows) if r["n_C"] == "10"]
base = T.baselines("A", [Pall[i] for i in dec], y[dec]) + T.baselines("B", Pall, y, z)
got = {(b[0], b[1]): float(b[3]) for b in base}
tex = open("main.tex").read()
tab = tex[tex.index(r"\label{tab:bp}"):]
lines = [l for l in tab[tab.index(r"\midrule"):tab.index(r"\bottomrule")].splitlines() if "&" in l]
pa = [float(x) for x in re.findall(r"\$([+-]?\d+\.\d{3})\$", lines[0])]
pb = [float(x) for x in re.findall(r"\$([+-]?\d+\.\d{3})\$", lines[1])]
for name, key, i in (("Base", "mean", 0), ("Class.", "classical best", 1), ("IRLA", "IRLA", 2), ("LO_1", "LO(0,0,1) fixed", 3), ("Ridge", "ridge on degree-pair counts", 4)):
    check(f"decanes {name}", pa[i], round(got[("A", key)], 3), 0.0006)
for name, key, i in (("Base", "size only", 0), ("Class.", "size + classical best", 1), ("IRLA", "size + IRLA", 2), ("LO_1", "size + LO(0,0,1) fixed", 3), ("Ridge", "ridge on degree-pair counts", 4)):
    check(f"pooled {name}", pb[i], round(got[("B", key)], 3), 0.0006)

OK       decanes Base: paper -0.062, recomputed -0.062
OK       decanes Class.: paper 0.176, recomputed 0.176
OK       decanes IRLA: paper 0.167, recomputed 0.167
OK       decanes LO_1: paper 0.149, recomputed 0.149
OK       decanes Ridge: paper 0.86, recomputed 0.86
OK       pooled Base: paper 0.938, recomputed 0.938
OK       pooled Class.: paper 0.967, recomputed 0.967
OK       pooled IRLA: paper 0.963, recomputed 0.963
OK       pooled LO_1: paper 0.963, recomputed 0.963
OK       pooled Ridge: paper 0.991, recomputed 0.991


In [5]:
if RUN_SLOW:
    run("cd external/bp && python bp_tests.py", tail=6)
s = pd.read_csv("external/bp/bp_results_summary.csv")
print(s.T)
for line, row, lab in ((pa, s.iloc[0], "decanes"), (pb, s.iloc[1], "pooled")):
    check(f"{lab} GM median (wide box)", line[5], round(row.median_Q2_GM12, 3), 0.0006)
    check(f"{lab} LO_h median", line[6], round(row.median_Q2_LOh, 3), 0.0006)
    check(f"{lab} median dQ2", line[7], round(row.median_dQ2_LOh_GM12, 3), 0.0006)
    check(f"{lab} pre-registered verdict", "no evidence that gamma adds value", row.prereg_verdict)

                                                     0  \
test                                     A: decane T_B   
n                                                   34   
median_Q2_GM2                                 0.468616   
median_Q2_LO2                                 0.464999   
median_Q2_GM12                                0.513711   
median_Q2_LOh                                 0.405868   
median_dQ2_LOh_GM12                           -0.07153   
LOh_better_of_50                                    15   
ci95_lo                                      -0.155843   
ci95_hi                                       0.001484   
prereg_verdict       no evidence that gamma adds value   

                                                        1  
test                 B: pooled C6-C10 T_B (size-adjusted)  
n                                                     100  
median_Q2_GM2                                    0.976376  
median_Q2_LO2                                    0.979976  
med

### The permutation check (selection-aware: the ridge penalty is re-chosen in every fold)

In [6]:
F = np.array([B.count_vector(Pall[i]) for i in dec], float); yd = y[dec]
real = B.scores(B.outer_loo(B.fold_ridge, F, yd)[0], yd)[0]
rng = np.random.default_rng(20261005); q = []
t0 = time.time()
for _ in range(200):
    yp = rng.permutation(yd); q.append(B.scores(B.outer_loo(B.fold_ridge, F, yp)[0], yp)[0])
q = np.array(q); print(f"{time.time() - t0:.0f} s")
check("observed ridge Q2, decanes", 0.860, round(real, 3), 0.0006)
check("permutation median", -0.09, round(float(np.median(q)), 2), 0.006)
check("permutation 95th percentile", -0.01, round(float(np.percentile(q, 95)), 2), 0.006)
check("permutation maximum", 0.20, round(float(q.max()), 2), 0.006)
check("permutations reaching 0.860", 0, int((q >= real).sum()))

248 s
OK       observed ridge Q2, decanes: paper 0.86, recomputed 0.86
OK       permutation median: paper -0.09, recomputed -0.09
OK       permutation 95th percentile: paper -0.01, recomputed -0.01
OK       permutation maximum: paper 0.2, recomputed 0.2
OK       permutations reaching 0.860: paper 0, recomputed 0


True

### Exploratory selection-aware resampling (added after the results; not pre-registered)

In [7]:
if RUN_SLOW:
    run("cd external/bp && python selection_resampling.py 500 200", tail=4)
sr = pd.read_csv("external/bp/selection_resampling_summary.csv")
print(sr)
check("decanes median dQ2", -0.110, round(sr.iloc[0].median_dQ2, 3), 0.0006)
check("decanes 2.5 percentile", -0.510, round(sr.iloc[0]["p2.5"], 3), 0.0006)
check("decanes 97.5 percentile", 0.312, round(sr.iloc[0]["p97.5"], 3), 0.0006)
check("pooled median dQ2", 0.001, round(sr.iloc[1].median_dQ2, 3), 0.0006)

                                   test  replicates  n_sub  median_dQ2  \
0                         A: decane T_B         500     27     -0.1097   
1  B: pooled C6-C10 T_B (size-adjusted)         200     80      0.0007   

     p2.5   p97.5  LOh_better  
0 -0.5097  0.3118         136  
1 -0.0146  0.0084         110  
OK       decanes median dQ2: paper -0.11, recomputed -0.11
OK       decanes 2.5 percentile: paper -0.51, recomputed -0.51
OK       decanes 97.5 percentile: paper 0.312, recomputed 0.312
OK       pooled median dQ2: paper 0.001, recomputed 0.001


True

## Summary

In [8]:
summary = pd.DataFrame(RESULTS)
display(summary) if "display" in dir() else print(summary.to_string())
n_bad = int((summary.status != "OK").sum())
print(f"{len(summary)} checks, {n_bad} mismatches")
assert n_bad == 0, "some recomputed values differ from the paper"

                             check                              paper                         recomputed status
0                     decanes Base                             -0.062                             -0.062     OK
1                   decanes Class.                              0.176                              0.176     OK
2                     decanes IRLA                              0.167                              0.167     OK
3                     decanes LO_1                              0.149                              0.149     OK
4                    decanes Ridge                               0.86                               0.86     OK
5                      pooled Base                              0.938                              0.938     OK
6                    pooled Class.                              0.967                              0.967     OK
7                      pooled IRLA                              0.963                              0.963